#### CHOOSING THE OPTIMAL NUMBER OF HIDDEN LAYERS, NEURONS AND ALL 

1) Simple Method -> Hit and Trial
2) Grid / Random Search
3) Cross Validation
4) Heuristics and Rules of Thumb

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
data=pd.read_csv('dataset/Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
encoder = OneHotEncoder(drop=None,sparse_output = False)
encoded_df=encoder.fit_transform(data[['Gender', 'Geography']])

In [6]:
encoded_df = pd.DataFrame(encoded_df,columns=encoder.get_feature_names_out(['Gender','Geography']))
encoded_df

,Gender_Female,Gender_Male,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,1.0,0.0,0.0
1,1.0,0.0,0.0,0.0,1.0
2,1.0,0.0,1.0,0.0,0.0
3,1.0,0.0,1.0,0.0,0.0
4,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...
9995,0.0,1.0,1.0,0.0,0.0
9996,0.0,1.0,1.0,0.0,0.0
9997,1.0,0.0,1.0,0.0,0.0
9998,0.0,1.0,0.0,1.0,0.0


In [7]:
data = pd.concat([data.drop(['Geography','Gender'],axis=1),encoded_df],axis=1)

In [8]:
data.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Gender_Female,Gender_Male,Geography_France,Geography_Germany,Geography_Spain
0,619,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,1.0,0.0,0.0
1,608,41,1,83807.86,1,0,1,112542.58,0,1.0,0.0,0.0,0.0,1.0
2,502,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,1.0,0.0,0.0
3,699,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,1.0,0.0,0.0
4,850,43,2,125510.82,1,1,1,79084.10,0,1.0,0.0,0.0,0.0,1.0


In [9]:
X = data.drop('Exited',axis=1)
y=data['Exited']

In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [11]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Creating a function to try different parameters (KerasClassifier)

In [12]:
def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

    return model


#### Create a Keras Clssifier

In [18]:
model=KerasClassifier(neurons=32,layers=1,build_fn=create_model,epochs=50,batch_size=10,verbose=0)

In [19]:
param_grid = {
    'neurons' : [16,32,64,128],
    'layers' : [1,2,3],
    'epochs' : [50,100]
}

In [20]:
grid = GridSearchCV(estimator=model, param_grid=param_grid,n_jobs=-1,cv=3)
grid_result = grid.fit(X_train,y_train)

c:\Projects\Bank-Retention-Analysis\venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Projects\Bank-Retention-Analysis\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
print(f"Best Params : {grid_result.best_score_,grid_result.best_params_}")

Best Params : (np.float64(0.8554666666666666), {'epochs': 50, 'layers': 1, 'neurons': 32})
